# Phase 8: nonuniform polarizations in complex dimension three

This notebook extends the exact relative-systole engine to ternary Hermitian polarizations on CM threefolds isogenous to $E_\Delta^3$. We validate common $g=3$ benchmarks, scan types $(1,1,2)$, $(1,1,3)$, and $(1,2,2)$, and then search the full 12-real-dimensional compatible-metric space for generic controls. Every claim below distinguishes an exact bounded-CM record from a numerical moduli-space record; neither is called a global optimum.

In [1]:
from pathlib import Path
from time import perf_counter
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from gkp_systole import (
    G3_NONUNIFORM_CM_BENCHMARKS, G3_TYPE_112_GAUSSIAN_FORM,
    G3_TYPE_113_EISENSTEIN_FORM, G3_TYPE_122_GAUSSIAN_FORM,
    ImaginaryQuadraticOrder, KLEIN_QUARTIC_PERIOD_MODEL,
    SystoleLedgerEntry, TernaryQuadraticHermitianForm,
    bounded_ternary_hermitian_forms, high_precision_coordinate_systole,
    refine_compatible_moduli_sample, refine_compatible_moduli_simplex,
    scan_compatible_moduli, ternary_hermitian_moduli_family,
    write_systole_ledger,
)

## Exact product and Klein-quartic checks

The diagonal Gaussian Hermitian forms are deliberately elementary checks of all conventions. The pre-existing Klein-quartic period model is the common non-product $g=3$, type $(2,2,2)$ benchmark.

In [2]:
product_forms = (
    TernaryQuadraticHermitianForm(ImaginaryQuadraticOrder(-4), 1, 1, 2),
    TernaryQuadraticHermitianForm(ImaginaryQuadraticOrder(-4), 1, 1, 3),
    TernaryQuadraticHermitianForm(ImaginaryQuadraticOrder(-4), 1, 2, 2),
)
expected_product = {(1, 1, 2): 0.5, (1, 1, 3): 1/3, (1, 2, 2): 0.5}
for form in product_forms:
    form.validate()
    core = form.compute_core_relative_systole()
    physical = float(core.squared_systole) / form.order.radicand**0.5
    assert abs(physical - expected_product[form.polarization_type]) < 1e-12
    print(form.polarization_type, 'product ell^2 =', physical,
          'multiplicity =', (core.class_multiplicity, core.lift_multiplicity))

klein = KLEIN_QUARTIC_PERIOD_MODEL.compute_qubit_systole()
assert klein.squared_systole_coefficient == 2
print('Klein quartic type (2,2,2): ell^2 =', klein.squared_systole,
      '= 1/sqrt(7), multiplicity =', (klein.class_multiplicity, klein.lift_multiplicity))

(1, 1, 2) product ell^2 = 0.5 multiplicity = (2, 4)
(1, 1, 3) product ell^2 = 0.3333333333333333 multiplicity = (4, 4)
(1, 2, 2) product ell^2 = 0.5 multiplicity = (4, 8)
Klein quartic type (2,2,2): ell^2 = 0.3779644730092272 = 1/sqrt(7), multiplicity = (21, 42)


## Exact Phase-8 CM records

The following representatives have exact integral metric cores. Validation checks the Riemann form, positivity, $J^2=-1$, determinant, polarization type, and kernel degree before solving every logical closest-vector problem.

In [3]:
cm_rows = []
for form in G3_NONUNIFORM_CM_BENCHMARKS:
    form.validate()
    result = form.compute_core_relative_systole()
    ell2 = float(result.squared_systole) / form.order.radicand**0.5
    cm_rows.append((form.polarization_type, form.order.discriminant,
                    (form.a, form.b, form.c), form.z12, form.z13, form.z23,
                    result.squared_systole, ell2,
                    result.class_multiplicity, result.lift_multiplicity))
for row in cm_rows:
    print(row)
assert abs(cm_rows[0][7] - 1.0) < 1e-12
assert abs(cm_rows[1][7] - 2/(3**0.5)) < 1e-12
assert abs(cm_rows[2][7] - 1.0) < 1e-12

((1, 1, 2), -4, (1, 2, 2), (0, 0), (0, 0), (-1, -1), Fraction(2, 1), 1.0, 3, 24)
((1, 1, 3), -3, (2, 2, 2), (-1, 0), (-1, 0), (0, 1), Fraction(2, 1), 1.1547005383792517, 8, 72)
((1, 2, 2), -4, (2, 2, 3), (0, 0), (-1, -1), (-1, -1), Fraction(2, 1), 1.0, 15, 60)


## Reproducible bounded ternary-Hermitian survey

We scan every imaginary-quadratic order discriminant $-40\leq\Delta<0$, with diagonal entries at most 5 and off-diagonal order-basis coefficients in $[-1,1]$. Mode permutations and diagonal unit changes are deduplicated exactly. This is a bounded candidate survey, not a complete classification under $GL(3,\mathcal O_\Delta)$.

In [4]:
discriminants = tuple(-n for n in range(3, 41) if (-n) % 4 in (0, 1))
survey_specs = ((2, (1,1,2)), (3, (1,1,3)), (4, (1,2,2)))
survey_results = {}
started = perf_counter()
for target_determinant, requested_type in survey_specs:
    records = []
    for discriminant in discriminants:
        forms = bounded_ternary_hermitian_forms(
            ImaginaryQuadraticOrder(discriminant), target_determinant,
            maximum_diagonal=5, off_diagonal_bound=1,
            requested_types=(requested_type,),
        )
        for form in forms:
            result = form.compute_core_relative_systole()
            ell2 = float(result.squared_systole) / form.order.radicand**0.5
            records.append((ell2, form, result))
    records.sort(key=lambda item: (-item[0], abs(item[1].order.discriminant), item[1]))
    survey_results[requested_type] = records
    best = records[0]
    print(requested_type, 'candidates =', len(records), 'best ell^2 =', best[0],
          'Delta =', best[1].order.discriminant, 'core =', best[2].squared_systole)
print('bounded survey seconds =', perf_counter() - started)
assert all(abs(survey_results[form.polarization_type][0][0] -
                   float(form.compute_core_relative_systole().squared_systole) /
                   form.order.radicand**0.5) < 1e-12
           for form in G3_NONUNIFORM_CM_BENCHMARKS)

(1, 1, 2) candidates = 1051 best ell^2 = 1.0 Delta = -4 core = 2


(1, 1, 3) candidates = 1070 best ell^2 = 1.1547005383792517 Delta = -3 core = 2


(1, 2, 2) candidates = 253 best ell^2 = 1.0 Delta = -4 core = 2
bounded survey seconds = 43.30154920788482


## Full 12-dimensional generic controls

For each exact CM record, we fix its integral polarization matrix and explore the entire compatible symmetric space (12 real coordinates). These samples are generic numerical controls; genericity makes CM overwhelmingly unlikely, but no arithmetic non-CM certificate is claimed for a floating-point sample.

In [5]:
searches = {}
started = perf_counter()
for index, form in enumerate(G3_NONUNIFORM_CM_BENCHMARKS):
    family = ternary_hermitian_moduli_family(form)
    search = scan_compatible_moduli(
        family, sample_count=600, seed=20260714 + index, radius=1.0,
        local_fraction=0.4, local_radius=0.18, refinement_starts=6,
        refinement_initial_step=0.05, refinement_minimum_step=0.003,
        refinement_rounds=16,
    )
    searches[form.polarization_type] = (family, search)
    print(form.polarization_type, 'CM =', family.reference_ell_squared,
          'best generic control =', search.best_sample.squared_systole,
          'samples beating CM =', search.number_beating_reference)
print('control-search seconds =', perf_counter() - started)

(1, 1, 2) CM = 1.0 best generic control = 1.0600499184566292 samples beating CM = 5


(1, 1, 3) CM = 1.1547005383792517 best generic control = 1.125471742723881 samples beating CM = 0


(1, 2, 2) CM = 1.0 best generic control = 0.9516345739886455 samples beating CM = 0
control-search seconds = 41.877614667173475


## Tighten and recheck the exceptional $(1,1,2)$ control

Only type $(1,1,2)$ produced controls above its bounded CM record. We refine that numerical point and recompute its systole at 60 decimal digits. The result remains a numerical moduli-space lower bound, not an exact algebraic reconstruction.

In [6]:
family112, search112 = searches[(1,1,2)]
refined112 = refine_compatible_moduli_sample(
    family112, search112.best_sample, seed=812, direction_count=40,
    initial_step=0.02, minimum_step=0.00025, maximum_rounds=36,
)
refined112 = refine_compatible_moduli_simplex(
    family112, refined112, initial_step=0.01, coordinate_tolerance=1e-6,
    value_tolerance=1e-11, maximum_iterations=600,
)
high_precision112 = high_precision_coordinate_systole(
    family112, refined112.coordinates, decimal_places=60,
)
print('refined double precision =', refined112.squared_systole)
print('60-digit recomputation    =', high_precision112)
print('multiplicity              =',
      (refined112.class_multiplicity, refined112.lift_multiplicity))
assert float(high_precision112) > 1.0
assert abs(float(high_precision112) - refined112.squared_systole) < 1e-11

refined double precision = 1.068957887238106
60-digit recomputation    = 1.06895788723810943986520288997919847944678284790231080541667
multiplicity              = (1, 2)


## Write the Phase-8 result ledger

The ledger records scope and arithmetic status explicitly. In particular, the $(1,1,2)$ generic control is not labeled CM or globally optimal.

In [7]:
exact_names = {(1,1,2): '1', (1,1,3): '2/sqrt(3)', (1,2,2): '1'}
entries = []
for form in G3_NONUNIFORM_CM_BENCHMARKS:
    result = form.compute_core_relative_systole()
    ell2 = float(result.squared_systole) / form.order.radicand**0.5
    entries.append(SystoleLedgerEntry(
        candidate_id=f'g3_cm_{"_".join(map(str, form.polarization_type))}', phase=8,
        dimension_g=3, polarization_type=str(form.polarization_type),
        family='ternary imaginary-quadratic Hermitian CM',
        cm_data=f'{form.order.label}; diag=({form.a},{form.b},{form.c}); z={form.z12,form.z13,form.z23}',
        ell_squared_decimal=f'{ell2:.16g}',
        ell_squared_exact=exact_names[form.polarization_type],
        class_multiplicity=result.class_multiplicity,
        lift_multiplicity=result.lift_multiplicity,
        metric_convention='polarization_scaled', arithmetic_status='exact certified',
        search_status='best in bounded Phase-8 CM survey',
        search_scope='|Delta|<=40, diagonal<=5, off-diagonal coefficients<=1',
    ))
entries.append(SystoleLedgerEntry(
    candidate_id='g3_generic_1_1_2_phase8', phase=8, dimension_g=3,
    polarization_type=str((1,1,2)), family='generic compatible metric control',
    cm_data='not classified; floating-point generic control',
    ell_squared_decimal=str(high_precision112), ell_squared_exact='not reconstructed',
    class_multiplicity=refined112.class_multiplicity,
    lift_multiplicity=refined112.lift_multiplicity,
    metric_convention='polarization_scaled',
    arithmetic_status='60-digit numerical recheck',
    search_status='best observed Phase-8 generic control',
    search_scope='600 Halton controls plus local direction/simplex refinement',
    notes='beats bounded CM record 1; no global or non-CM certificate',
))
write_systole_ledger(entries, json_path=ROOT/'data/phase8_result_ledger.json',
                      csv_path=ROOT/'data/phase8_result_ledger.csv')
print('wrote', len(entries), 'Phase-8 ledger entries')

wrote 4 Phase-8 ledger entries


## Conclusions

1. The exact $g=3$ engine and common product/Klein benchmarks agree with known formulas.
2. Coupled CM polarizations substantially improve all three product baselines in the bounded survey.
3. The bounded CM records remain unbeaten by this generic scan for $(1,1,3)$ and $(1,2,2)$.
4. For $(1,1,2)$ a generic compatible metric beats the bounded CM record, so the current data do **not** support declaring a CM optimizer for that type. The next step is exact reconstruction of the numerical control, followed by a broader CM search and independent multi-start moduli search.